# Code_1 — Data, Schema Registry and Benchmarks

**GPU: no** (CPU runtime is enough)

What this notebook produces (all saved to `data/<data_version>/` on Drive):

| File | Contents |
|---|---|
| `registry.json` | Schema registry: every function's parameters with canonical type, required flag, enum/min/max |
| `benchmark_single.json` | Single-error benchmark (clean calls + one injected error per call) |
| `benchmark_multi.json` | Multi-error benchmark (1, 2, 3, 4 errors per call), functions unseen in tuning |
| `splits.json` | Random stratified split and function-disjoint split of the single-error benchmark |
| `real_eval.json` | ToolAlpaca evaluation queries with gold calls, for real LLM generation (Code_5) |
| `examples_by_type.json` | One before/after example per error type (for the paper) |
| `data_summary.json` | All dataset statistics printed below |

Run order: **Code_1 → Code_2 → Code_3 → Code_4 → Code_5 → Code_6**, each on a fresh runtime.
Re-run this notebook only if `data_version` in `config.json` changes.

## Setup — Drive, shared code and config

Mounts Drive, imports `toolhall_core.py` from `hallucination_project_v2/code/`, and reads `config.json`.
Before the first run, upload **`toolhall_core.py`** and **`config.json`** to that folder.

In [ ]:
import os, sys, json, importlib, subprocess, random, copy, re, ast, time
from collections import Counter, defaultdict

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hallucination_project_v2'
    RAW = '/content/raw'
else:
    DRIVE_ROOT = os.environ.get('TOOLHALL_ROOT', os.path.abspath('hallucination_project_v2'))
    RAW = os.environ.get('TOOLHALL_RAW', os.path.abspath('raw'))
CODE_DIR = os.path.join(DRIVE_ROOT, 'code')
if not os.path.exists(os.path.join(CODE_DIR, 'toolhall_core.py')):
    raise FileNotFoundError(f"Upload toolhall_core.py and config.json to {CODE_DIR} first.")
sys.path.insert(0, CODE_DIR)
import toolhall_core as th
importlib.reload(th)

NOTEBOOK = 'Code_1'
CFG = th.load_config(os.path.join(CODE_DIR, 'config.json'))
CFG['drive_root'] = DRIVE_ROOT
P = th.Paths(CFG)
SEED = CFG['seed']
th.set_seed(SEED)
os.makedirs(RAW, exist_ok=True)
print(f"core v{th.CORE_VERSION} | data_version={CFG['data_version']} | seed={SEED}")
print(f"data folder: {P.data}")

## Download the three source datasets

API-Bank (2 files from Hugging Face), ToolAlpaca (2 files from GitHub) and ToolBench (Hugging Face dataset).
They are downloaded to the runtime's local disk, not to Drive.

In [ ]:
def sh(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"command failed: {cmd}")

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)

import urllib.request

SOURCES = {
    os.path.join(RAW, 'api-bank', 'training-data', 'lv1-train.json'):
        'https://huggingface.co/datasets/liminghao1630/API-Bank/resolve/main/training-data/lv1-train.json',
    os.path.join(RAW, 'api-bank', 'training-data', 'lv2-train.json'):
        'https://huggingface.co/datasets/liminghao1630/API-Bank/resolve/main/training-data/lv2-train.json',
    os.path.join(RAW, 'toolalpaca', 'data', 'eval_real.json'):
        'https://raw.githubusercontent.com/tangqiaoyu/ToolAlpaca/main/data/eval_real.json',
    os.path.join(RAW, 'toolalpaca', 'data', 'eval_simulated.json'):
        'https://raw.githubusercontent.com/tangqiaoyu/ToolAlpaca/main/data/eval_simulated.json',
}


def is_json_file(path):
    """True if the file exists and parses as JSON (rejects Git LFS pointer files)."""
    try:
        with open(path, encoding='utf-8') as f:
            json.load(f)
        return True
    except Exception:
        return False


for path, url in SOURCES.items():
    if is_json_file(path):
        print(f"ok (cached)  {os.path.relpath(path, RAW)}")
        continue
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for attempt in range(3):
        try:
            urllib.request.urlretrieve(url, path)
            if is_json_file(path):
                break
        except Exception as e:
            print(f"  download attempt {attempt + 1} failed: {e}")
        time.sleep(3)
    if not is_json_file(path):
        raise RuntimeError(f"Could not download valid JSON for {path} from {url}")
    print(f"downloaded   {os.path.relpath(path, RAW)}  ({os.path.getsize(path) // 1024:,} KB)")


def load_toolbench_rows():
    """ToolBench rows as dicts with api_list (string) and query."""
    local = os.path.join(RAW, 'toolbench')
    files = sorted(f for f in os.listdir(local) if f.endswith('.parquet')) if os.path.isdir(local) else []
    if files:
        import pyarrow.parquet as pq
        rows = []
        for f in files:
            rows += pq.read_table(os.path.join(local, f),
                                  columns=['api_list', 'query', 'query_id']).to_pylist()
        return rows
    from datasets import load_dataset
    ds = load_dataset("Maurus/ToolBench")["train"]
    return [{'api_list': r['api_list'], 'query': r['query'], 'query_id': r['query_id']} for r in ds]

toolbench_rows = load_toolbench_rows()
print(f"ToolBench rows: {len(toolbench_rows):,}")

## Parse the sources into clean reference calls and one schema registry

* **ToolBench** — the call is the first API of each row with every declared parameter filled
  with its published example value (`default`). The registry records the declared type
  (mapped to 6 canonical types), and `required` is **true only for `required_parameters`**
  (the previous version marked optional parameters required too).
  Example values are cast losslessly to their declared type (`"2"` → `2` for NUMBER) when
  `benchmark.cast_reference_values` is true; ToolBench stores almost all example values as strings.
  Example values are **not** stored in the registry: the reference calls are built from them,
  so using them for correction would be circular.
* **API-Bank** — the `input` field holds one or more API description JSON objects **followed by
  the dialogue**. All objects are parsed into the registry and the dialogue becomes the query
  (the previous version could not parse this field, so API-Bank schemas were missing and the
  query held only the generic instruction).
* **ToolAlpaca eval sets** — kept apart for the real-LLM evaluation; their OpenAPI documents
  are parsed into the registry (types, required flags, enums, min/max, defaults).

In [ ]:
CAST_REF = CFG['benchmark'].get('cast_reference_values', True)
registry = {}
reg_conflicts = Counter()
cast_stats = Counter()


_variants = defaultdict(dict)
_variant_counts = defaultdict(Counter)
_name_source = {}


def add_function(name, entry):
    """Collect every definition. A function defined several times with DIFFERENT parameter
    lists (common in API-Bank, where the same apiCode appears in many API versions) keeps its
    MOST FREQUENT definition (see finalize_registry); reference calls written against another
    variant are removed later by the structural-consistency filter."""
    if name in _name_source and _name_source[name] != entry['source']:
        reg_conflicts[f"name clash {_name_source[name]} vs {entry['source']} (first kept)"] += 1
        return
    _name_source[name] = entry['source']
    sig = tuple(sorted(entry['parameters']))
    _variants[name].setdefault(sig, entry)
    _variant_counts[name][sig] += 1


def finalize_registry():
    for name, cnt in _variant_counts.items():
        sig, _ = max(cnt.items(), key=lambda kv: (kv[1], -len(kv[0]), kv[0]))
        registry[name] = _variants[name][sig]
        if len(cnt) > 1:
            reg_conflicts[f"functions with several schema variants ({_name_source[name]})"] += 1


def parse_toolbench_row(row):
    al = row['api_list']
    try:
        al = ast.literal_eval(al) if isinstance(al, str) else al
    except Exception:
        return None
    if not al or not isinstance(al, list) or not isinstance(al[0], dict):
        return None
    api = al[0]
    if not api.get('api_name'):
        return None
    fn = f"{api.get('tool_name', '')}.{api['api_name']}" if api.get('tool_name') else api['api_name']
    req = [p for p in (api.get('required_parameters') or []) if isinstance(p, dict) and p.get('name')]
    opt = [p for p in (api.get('optional_parameters') or []) if isinstance(p, dict) and p.get('name')]
    pspec, call_params = {}, {}
    for p, is_req in [(p, True) for p in req] + [(p, False) for p in opt]:
        name = p['name']
        ctype = th.canonical_type(p.get('type'))
        pspec[name] = {'type': ctype, 'type_raw': p.get('type'), 'required': is_req,
                       'description': (p.get('description') or '')[:300]}
        if 'example_value' in p:
            v = p['example_value']
        elif p.get('default') is not None:
            v = p['default']
        elif ctype in ('integer', 'number'):
            v = 1
        elif ctype == 'boolean':
            v = True
        else:
            v = f"example_{name}"
        if CAST_REF:
            v2, was = th.cast_reference_value(v, ctype)
            cast_stats['cast' if was else 'kept'] += 1
            v = v2
        call_params[name] = v
    entry = {'description': (api.get('api_description') or '').strip()[:500],
             'source': 'toolbench', 'parameters': pspec}
    return fn, entry, {'function': fn, 'parameters': call_params}


pool = []
tb_stats = Counter()
for i, row in enumerate(toolbench_rows):
    r = parse_toolbench_row(row)
    if r is None:
        tb_stats['unparseable'] += 1
        continue
    fn, entry, call = r
    add_function(fn, entry)
    pool.append({'base_id': f"tb{i}", 'source': 'toolbench', 'query': (row.get('query') or '').strip(),
                 'tool_description': entry['description'], 'ground_truth_tool_call': call})
    tb_stats['parsed'] += 1
print(f"ToolBench: {dict(tb_stats)} | reference values cast: {dict(cast_stats)}")

_dec = json.JSONDecoder()


def split_apibank_input(s):
    """-> (list of API description dicts, dialogue text)."""
    apis, pos = [], 0
    s = s or ''
    while True:
        while pos < len(s) and s[pos].isspace():
            pos += 1
        if pos >= len(s) or s[pos] != '{':
            break
        try:
            obj, end = _dec.raw_decode(s, pos)
        except json.JSONDecodeError:
            break
        if isinstance(obj, dict):
            apis.append(obj)
        pos = end
    return apis, s[pos:].strip()


ab_parse_stats = Counter()


def parse_apibank_call(out):
    """[ApiName(key1='v1', key2={...})] -> call.  Parsed with Python's ast so dict / list
    values survive (the previous regex cut "{'a': 1}" down to "{"); regex only as fallback."""
    raw = str(out or '').replace('API-Request:', '').strip()
    m = re.match(r'\[(\w+)\((.*)\)\]', raw, re.DOTALL)
    if not m:
        return None
    params = {}
    try:
        node = ast.parse(f"f({m.group(2)})", mode='eval').body
        for kw in node.keywords:
            if kw.arg is None:
                raise ValueError('**kwargs')
            params[kw.arg] = ast.literal_eval(kw.value)
        ab_parse_stats['ast'] += 1
    except Exception:
        params = {}
        for mm in re.finditer(r"(\w+)\s*=\s*'([^']*)'|(\w+)\s*=\s*\"([^\"]*)\"", m.group(2).strip()):
            if mm.group(1):
                params[mm.group(1)] = mm.group(2)
            elif mm.group(3):
                params[mm.group(3)] = mm.group(4)
        ab_parse_stats['regex_fallback'] += 1
    return {'function': m.group(1), 'parameters': params}


ab_stats = Counter()
for fname in ['lv1-train.json', 'lv2-train.json']:
    path = os.path.join(RAW, 'api-bank', 'training-data', fname)
    with open(path, encoding='utf-8') as f:
        items = json.load(f)
    for j, it in enumerate(items):
        out = it.get('output', it.get('expected_output'))
        call = parse_apibank_call(out)
        apis, dialogue = split_apibank_input(it.get('input', ''))
        for a in apis:
            code = a.get('apiCode')
            if not code:
                continue
            ps = {}
            for pn, spec in (a.get('parameters') or {}).items():
                spec = spec if isinstance(spec, dict) else {}
                ps[pn] = {'type': th.canonical_type(spec.get('type')), 'type_raw': spec.get('type'),
                          'required': spec.get('required') if isinstance(spec.get('required'), bool) else None,
                          'description': (spec.get('description') or '')[:300]}
            add_function(code, {'description': (a.get('description') or '')[:500],
                                'source': 'apibank', 'parameters': ps})
        if call is None:
            ab_stats['not_an_api_call'] += 1
            continue
        pool.append({'base_id': f"ab_{fname[:3]}_{j}", 'source': 'apibank', 'query': dialogue,
                     'tool_description': next((a.get('description', '') for a in apis
                                               if a.get('apiCode') == call['function']), ''),
                     'ground_truth_tool_call': call})
        ab_stats['parsed'] += 1
print(f"API-Bank: {dict(ab_stats)} | call parsing: {dict(ab_parse_stats)}")

def _resolve(doc, node, depth=0):
    while isinstance(node, dict) and '$ref' in node and depth < 12:
        ref = node['$ref']
        if not isinstance(ref, str) or not ref.startswith('#/'):
            return {}
        cur = doc
        for part in ref[2:].split('/'):
            cur = cur.get(part, {}) if isinstance(cur, dict) else {}
        node, depth = cur, depth + 1
    return node if isinstance(node, dict) else {}


def _spec_from_schema(sch, required, description):
    spec = {'type': th.canonical_type(sch.get('type')), 'type_raw': sch.get('type'),
            'required': bool(required), 'description': (description or sch.get('description') or '')[:300]}
    if isinstance(sch.get('enum'), list) and sch['enum']:
        spec['enum'] = sch['enum']
    for k in ('minimum', 'maximum', 'default'):
        if k in sch:
            spec[k] = sch[k]
    return spec


def openapi_params(doc, path, method):
    path_item = (doc.get('paths') or {}).get(path) or {}
    op = path_item.get(str(method).lower()) or {}
    if not op:
        return None, ''
    out = {}
    for prm in (path_item.get('parameters') or []) + (op.get('parameters') or []):
        prm = _resolve(doc, prm)
        if not prm.get('name'):
            continue
        sch = _resolve(doc, prm.get('schema') or {})
        out[prm['name']] = _spec_from_schema(sch, prm.get('required', prm.get('in') == 'path'),
                                             prm.get('description'))
    body = _resolve(doc, op.get('requestBody') or {})
    content = body.get('content') or {}
    ctype = 'application/json' if 'application/json' in content else (next(iter(content), None))
    if ctype:
        sch = _resolve(doc, (content.get(ctype) or {}).get('schema') or {})
        req = set(sch.get('required') or [])
        for name, ps in (sch.get('properties') or {}).items():
            out[name] = _spec_from_schema(_resolve(doc, ps), name in req, None)
    return out, (op.get('summary') or op.get('description') or '')


def params_from_function_description(text):
    """Fallback: ToolAlpaca's 'Parameters: {...}' text."""
    m = re.search(r'Parameters:\s*(\{.*?\})\s*\n', (text or '') + '\n', re.DOTALL)
    if not m:
        return {}
    try:
        d = json.loads(m.group(1))
    except Exception:
        return {}
    out = {}
    for name, desc in d.items():
        desc = str(desc)
        req = desc.strip().lower().startswith('required')
        tm = re.search(r'(string|integer|number|boolean|array|object)', desc.lower())
        spec = {'type': th.canonical_type(tm.group(1)) if tm else None, 'type_raw': tm.group(1) if tm else None,
                'required': req, 'description': desc[:300]}
        em = re.search(r'One of: \[([^\]]*)\]', desc)
        if em:
            spec['enum'] = [x.strip() for x in em.group(1).split(',') if x.strip()]
        out[name] = spec
    return out


real_eval, ta_stats = [], Counter()
for fname in ['eval_real.json', 'eval_simulated.json']:
    with open(os.path.join(RAW, 'toolalpaca', 'data', fname), encoding='utf-8') as f:
        tools = json.load(f)
    for tool in tools:
        try:
            doc = json.loads(tool.get('Documentation') or '{}')
        except Exception:
            doc = {}
        fixed_paths = {}
        for pth, item in (doc.get('paths') or {}).items():
            if not str(pth).startswith('/') or not isinstance(item, dict):
                continue
            fixed_paths.setdefault(pth, {})
            for key, val in item.items():
                if str(key).startswith('/') and isinstance(val, dict):
                    fixed_paths.setdefault(key, {}).update(val)
                    ta_stats['nested_path_recovered'] += 1
                else:
                    fixed_paths[pth][key] = val
        doc['paths'] = fixed_paths
        proj = dict(tool.get('Function_Projection') or {})
        fdesc = {k: v for k, v in (tool.get('Function_Description') or {}).items() if k != 'components'}
        for path, item in (doc.get('paths') or {}).items():
            for method, op in (item or {}).items():
                if method.lower() not in ('get', 'post', 'put', 'patch', 'delete') or not isinstance(op, dict):
                    continue
                name = op.get('operationId') or (path.strip('/').replace('/', '_') + '_' + method.lower())
                proj.setdefault(name, [path, method])
        tool_functions = []
        for fn_name in list(dict.fromkeys(list(fdesc) + list(proj))):
            params, summary = (None, '')
            if fn_name in proj and isinstance(proj[fn_name], list) and len(proj[fn_name]) >= 2:
                params, summary = openapi_params(doc, proj[fn_name][0], proj[fn_name][1])
            how = 'openapi' if fn_name in fdesc else 'openapi_added'
            if params is None:
                params, how = params_from_function_description(fdesc.get(fn_name, '')), 'description_text'
            ta_stats[how] += 1
            desc = (str(fdesc.get(fn_name, '')).split('\nParameters:')[0] or summary)[:500]
            add_function(fn_name, {'description': desc, 'source': 'toolalpaca_eval',
                                   'tool': tool.get('Name', ''), 'parameters': params})
            tool_functions.append(fn_name)
        for k, (instr, gold) in enumerate(zip(tool.get('Instructions', []), tool.get('Golden_Answers', []))):
            if not gold or not isinstance(gold, list) or not isinstance(gold[0], dict):
                ta_stats['no_gold'] += 1
                continue
            act = gold[0]
            ai = act.get('Action_Input', {})
            if isinstance(ai, str):
                try:
                    ai = json.loads(ai) if ai.strip() else {}
                except Exception:
                    ai = {}
            if not act.get('Action'):
                ta_stats['no_action'] += 1
                continue
            real_eval.append({'id': f"R{len(real_eval):04d}", 'dataset': fname.replace('.json', ''),
                              'tool_name': tool.get('Name', ''), 'query': instr,
                              'tool_functions': tool_functions,
                              'gt_call': {'function': act['Action'], 'parameters': ai if isinstance(ai, dict) else {}},
                              'n_gold_steps': len(gold),
                              'gold_is_clarification': act['Action'] == 'getDetails'})
finalize_registry()
print(f"ToolAlpaca eval: {len(real_eval)} queries | schema parsing: {dict(ta_stats)}")
print(f"Registry: {len(registry):,} functions | {dict(reg_conflicts)}")

## Registry statistics

In [ ]:
REG = th.Registry(registry, CFG['validator']['unknown_required_policy'])
reg_stats = {'functions_by_source': dict(Counter(e['source'] for e in registry.values())),
             'declarations': 0, 'canonical_type_known': 0, 'required_true': 0,
             'required_false': 0, 'required_unknown': 0, 'with_enum': 0,
             'raw_types': Counter(), 'canonical_types': Counter()}
for e in registry.values():
    for pn, s in e['parameters'].items():
        reg_stats['declarations'] += 1
        reg_stats['canonical_type_known'] += int(s.get('type') is not None)
        r = s.get('required')
        reg_stats['required_true' if r is True else 'required_false' if r is False else 'required_unknown'] += 1
        reg_stats['with_enum'] += int(bool(s.get('enum')))
        reg_stats['raw_types'][str(s.get('type_raw'))] += 1
        reg_stats['canonical_types'][str(s.get('type'))] += 1
reg_stats['raw_types'] = dict(reg_stats['raw_types'].most_common(25))
reg_stats['canonical_types'] = dict(reg_stats['canonical_types'])
print(json.dumps(reg_stats, indent=1))

## Build the clean-call pool

Kept: ToolBench and API-Bank calls with at least one parameter whose function **is in the
registry** (a schema-based method cannot be evaluated on calls that have no schema — the
previous benchmark had 62 such clean calls, all counted as false positives).

A reference call must also be **structurally consistent with its own schema**: no undeclared
parameter names and no missing required parameter (some API-Bank training answers use other
parameter names than their API description, e.g. `date_from` where the schema says `start_date`).
Such calls cannot serve as "correct" references and are removed and counted. **Type**
mismatches are kept on purpose: they are the real-world inconsistency between published types
and published values that the adaptive type check is designed for.

At most `max_calls_per_function` calls per function, so a few popular APIs cannot dominate.

In [ ]:
rng = random.Random(SEED)
cap = CFG['benchmark']['max_calls_per_function']
pool_stats = Counter()
eligible = []
_struct = th.Validator(REG, {'check_type': False, 'check_value': False},
                       CFG['envelope']['accepted_key_sets'])
for b in pool:
    call = b['ground_truth_tool_call']
    fk = REG.resolve(call['function'])
    if fk is None:
        pool_stats['function_not_in_registry'] += 1
        continue
    if not call['parameters']:
        pool_stats['no_parameters'] += 1
        continue
    bad = _struct.validate(call)
    if bad:
        pool_stats[f"reference_inconsistent_with_schema/{b['source']}"] += 1
        continue
    b['function_key'] = fk
    eligible.append(b)
eligible.sort(key=lambda b: b['base_id'])
rng.shuffle(eligible)
per_func, base_pool = Counter(), []
seen_calls = set()
for b in eligible:
    sig = (th.canonical_json(b['ground_truth_tool_call']), b['query'])
    if sig in seen_calls:
        pool_stats['duplicate'] += 1
        continue
    if per_func[b['function_key']] >= cap:
        pool_stats['over_function_cap'] += 1
        continue
    seen_calls.add(sig)
    per_func[b['function_key']] += 1
    base_pool.append(b)
pool_stats['kept'] = len(base_pool)
pool_stats['distinct_functions'] = len(per_func)
print(dict(pool_stats))
print('by source:', dict(Counter(b['source'] for b in base_pool)))

## Inject labelled errors — single-error benchmark

`n_clean` clean calls plus `n_per_type` calls for each of the six error types, each built from a
**different** clean call (the previous version reused the same base calls across types).
Clean calls are drawn first so they are a random sample of the pool; then the types with the
strictest eligibility (WRONG_TYPE, MISSING_PARAMETER, WRONG_VALUE) are filled.

In [ ]:
donors = th.Injector.build_donor_index([b['ground_truth_tool_call'] for b in base_pool])
inj = th.Injector(REG, random.Random(SEED + 1), donors, CFG['benchmark']['wrong_value_mode'])
N_CLEAN, N_TYPE = CFG['benchmark']['n_clean'], CFG['benchmark']['n_per_type']


def make_example(ex_id, b, generated, types, metas):
    return {'id': ex_id, 'base_id': b['base_id'], 'source': b['source'],
            'function_key': b['function_key'], 'query': b['query'],
            'tool_description': b['tool_description'][:300],
            'ground_truth_tool_call': b['ground_truth_tool_call'],
            'generated_tool_call': generated,
            'error_types': sorted(types, key=th.PRIMARY_ORDER.index),
            'hallucination_type': 'NONE' if not types else (types[0] if len(types) == 1 else f"MULTI_{len(types)}"),
            'is_hallucinated': bool(types), 'injection': metas,
            'n_params': len(b['ground_truth_tool_call']['parameters'])}


used = set()
single = []
for b in base_pool:
    if len([e for e in single if not e['error_types']]) >= N_CLEAN:
        break
    used.add(b['base_id'])
    single.append(make_example(f"S{len(single):05d}", b, copy.deepcopy(b['ground_truth_tool_call']), [], []))

inj_stats = {}
for t in ['WRONG_TYPE', 'MISSING_PARAMETER', 'WRONG_VALUE', 'EXTRA_PARAMETER', 'WRONG_FUNCTION', 'WRONG_FORMAT']:
    got, tried, reasons = 0, 0, Counter()
    for b in base_pool:
        if got >= N_TYPE:
            break
        if b['base_id'] in used:
            continue
        tried += 1
        gen, meta = inj.inject(b['ground_truth_tool_call'], [t], b['query'])
        if gen is None:
            reasons[meta.split(':')[-1].strip()] += 1
            continue
        used.add(b['base_id'])
        single.append(make_example(f"S{len(single):05d}", b, gen, [t], meta))
        got += 1
    inj_stats[t] = {'made': got, 'tried': tried, 'skipped_reasons': dict(reasons)}
    print(f"{t:<18} {got}/{N_TYPE}  (tried {tried}; skipped: {dict(reasons)})")

bad = [e['id'] for e in single if e['is_hallucinated']
       and th.calls_equal(e['generated_tool_call'], e['ground_truth_tool_call'])
       and e['error_types'] != ['WRONG_FORMAT']]
assert not bad, f"injections that changed nothing: {bad[:5]}"
print(f"\nsingle-error benchmark: {len(single)} examples  {dict(Counter(e['hallucination_type'] for e in single))}")
wv = [e for e in single if e['error_types'] == ['WRONG_VALUE']]
print('WRONG_VALUE methods:', dict(Counter(e['injection'][0]['method'].split(':')[0] for e in wv)),
      '| original value grounded in query:',
      dict(Counter(str(e['injection'][0]['gt_value_grounded']) for e in wv)))

## Splits — random stratified and function-disjoint

* **random**: stratified 30/70 by label (as before). The same function may appear in both parts.
* **function_disjoint**: every function is in tune OR test, never both, so test functions are unseen
  during tuning (#1). Both splits are saved; Code_2–Code_4 report results on both test parts.

In [ ]:
tf = CFG['benchmark']['tune_frac']
r_tune, r_test = th.split_random(single, tf, SEED)
d_tune, d_test, tune_funcs = th.split_function_disjoint(single, tf, SEED)
splits = {'random': {'tune': r_tune, 'test': r_test, 'report': th.split_report(single, r_tune, r_test)},
          'function_disjoint': {'tune': d_tune, 'test': d_test, 'tune_functions': tune_funcs,
                                'report': th.split_report(single, d_tune, d_test)}}
for k, v in splits.items():
    print(f"\n[{k}]")
    print(json.dumps(v['report'], indent=1))
assert splits['function_disjoint']['report']['functions_in_both'] == 0

## Multi-error benchmark (#2)

Calls with exactly k = 1, 2, 3, 4 errors of **different** types, built only from functions that are
in the function-disjoint **test** side (so the benchmark is valid for both splits). The k error
types are drawn at random among the combinations that can be applied to the call; errors never
touch the same parameter twice. This benchmark is used only for testing.

In [ ]:
from itertools import combinations
tune_func_set = set(tune_funcs)
multi = []
multi_stats = {}
rng_m = random.Random(SEED + 2)
remaining = [b for b in base_pool if b['base_id'] not in used and b['function_key'] not in tune_func_set]
print(f"pool for multi-error: {len(remaining)} unused calls on test-side functions")
for k in CFG['benchmark']['multi_error_k']:
    combos = list(combinations(th.ERROR_TYPES, k))
    got, tried = 0, 0
    for b in remaining:
        if got >= CFG['benchmark']['n_per_k']:
            break
        if b['base_id'] in used:
            continue
        tried += 1
        order = combos[:]
        rng_m.shuffle(order)
        for combo in order:
            gen, meta = inj.inject(b['ground_truth_tool_call'], list(combo), b['query'])
            if gen is not None:
                used.add(b['base_id'])
                multi.append(make_example(f"M{k}_{got:04d}", b, gen, list(combo), meta))
                got += 1
                break
    multi_stats[k] = {'made': got, 'tried': tried,
                      'type_counts': dict(Counter(t for e in multi if len(e['error_types']) == k
                                                  for t in e['error_types']))}
    print(f"k={k}: {got} examples (tried {tried})  type counts {multi_stats[k]['type_counts']}")

## Real-LLM evaluation set (ToolAlpaca) — known vs unseen tools

The 21 ToolAlpaca evaluation tools are split (seeded) into **known** and **unseen** tools.
For known tools, part of their gold calls is reserved for calibration (these are the only real
calls the system may learn from); the rest, and all unseen-tool queries, are evaluated in Code_5.

In [ ]:
tools = sorted({r['tool_name'] for r in real_eval})
rng_r = random.Random(SEED + 3)
rng_r.shuffle(tools)
n_known = int(round(CFG['real_eval']['known_tool_frac'] * len(tools)))
known_tools = set(tools[:n_known])
for r in sorted(real_eval, key=lambda r: r['id']):
    r['tool_group'] = 'known' if r['tool_name'] in known_tools else 'unseen'
    r['role'] = 'excluded_clarification' if r['gold_is_clarification'] else 'eval'
for t in sorted(known_tools):
    rows = sorted([r for r in real_eval if r['tool_name'] == t and r['role'] == 'eval'],
                  key=lambda r: r['id'])
    rng_r.shuffle(rows)
    n_cal = int(round(CFG['real_eval']['known_calibration_frac'] * len(rows)))
    for r in rows[:n_cal]:
        r['role'] = 'calibration'
gt_in_reg = Counter(REG.resolve(r['gt_call']['function']) is not None for r in real_eval)
real_stats = {'n_queries': len(real_eval), 'n_tools': len(tools), 'known_tools': sorted(known_tools),
              'by_group_role': dict(Counter(f"{r['tool_group']}/{r['role']}" for r in real_eval)),
              'gold_function_in_registry': dict(gt_in_reg),
              'multi_step_gold': sum(1 for r in real_eval if r['n_gold_steps'] > 1)}
print(json.dumps(real_stats, indent=1))
_v = th.Validator(REG, CFG['validator'], CFG['envelope']['accepted_key_sets'])
gold_viol = Counter(v['type'] for r in real_eval if not r['gold_is_clarification']
                    for v in _v.validate(r['gt_call']))
real_stats['gold_call_violations_strict_types'] = dict(gold_viol)
print('violations in GOLD calls (strict type checking):', dict(gold_viol))

## One example per error type (before / after)

In [ ]:
examples_by_type = {}
test_set = set(splits['random']['test'])
for i, e in enumerate(single):
    lab = e['hallucination_type']
    if lab in examples_by_type or i not in test_set:
        continue
    examples_by_type[lab] = {'id': e['id'], 'query': e['query'][:300],
                             'reference_call': e['ground_truth_tool_call'],
                             'generated_call': e['generated_tool_call'], 'injection': e['injection']}
for e in multi:
    lab = e['hallucination_type']
    if lab.startswith('MULTI') and lab not in examples_by_type:
        examples_by_type[lab] = {'id': e['id'], 'query': e['query'][:300],
                                 'reference_call': e['ground_truth_tool_call'],
                                 'generated_call': e['generated_tool_call'], 'injection': e['injection']}
for lab in th.ALL_LABELS + ['MULTI_2', 'MULTI_3', 'MULTI_4']:
    if lab in examples_by_type:
        x = examples_by_type[lab]
        print(f"\n[{lab}]  {x['id']}")
        print('  reference :', json.dumps(x['reference_call'], ensure_ascii=False)[:220])
        print('  generated :', json.dumps(x['generated_call'], ensure_ascii=False)[:220])
        if x['injection']:
            print('  injection :', json.dumps(x['injection'], ensure_ascii=False, default=str)[:220])

## Save everything to Drive

In [ ]:
summary = {
    'registry': reg_stats, 'toolbench': dict(tb_stats), 'apibank': dict(ab_stats),
    'reference_value_casting': dict(cast_stats), 'cast_reference_values': CAST_REF,
    'pool': dict(pool_stats), 'pool_by_source': dict(Counter(b['source'] for b in base_pool)),
    'single': {'n': len(single), 'by_label': dict(Counter(e['hallucination_type'] for e in single)),
               'by_source': dict(Counter(e['source'] for e in single)),
               'injection': inj_stats,
               'avg_params_by_label': {l: round(sum(e['n_params'] for e in single if e['hallucination_type'] == l)
                                                / max(1, sum(1 for e in single if e['hallucination_type'] == l)), 2)
                                       for l in th.ALL_LABELS}},
    'multi': {'n': len(multi), 'by_k': multi_stats},
    'splits': {k: v['report'] for k, v in splits.items()},
    'real_eval': real_stats, 'taxonomy': th.TAXONOMY,
    'hardware': th.hardware_info(), 'packages': th.package_versions(),
}
th.save_json(registry, P.data_file('registry.json'), CFG, NOTEBOOK, kind='data')
th.save_json(single, P.data_file('benchmark_single.json'), CFG, NOTEBOOK, kind='data')
th.save_json(multi, P.data_file('benchmark_multi.json'), CFG, NOTEBOOK, kind='data')
th.save_json(splits, P.data_file('splits.json'), CFG, NOTEBOOK, kind='data')
th.save_json(real_eval, P.data_file('real_eval.json'), CFG, NOTEBOOK, kind='data')
th.save_json(examples_by_type, P.data_file('examples_by_type.json'), CFG, NOTEBOOK, kind='data')
th.save_json(summary, P.data_file('data_summary.json'), CFG, NOTEBOOK, kind='data')
for f in sorted(os.listdir(P.data)):
    print(f"  {f:<28} {os.path.getsize(os.path.join(P.data, f)) // 1024:>7} KB")
print("\nCode_1 complete.")